# Dunnhumby M5: M3 두 축 그래프 + M4 손실 두 변형 (개발 시드 43)

두 arm 모두 확증된 **M3 두 축(V+N) 중심화 가치그래프를 그대로** 쓰고, **BPR 학습행 가중치만** 다릅니다.

- **A (원형 M4)**: `1 + 0.5·q_C(u)·상품금액백분위(i)·고객가격대적합도(u,i)` — H&M test에서 확인된 손실
- **B (N/V 분리)**: `(1 + 0.5·q_N(u)) × (1 + 0.5·q_V(u)·상품금액백분위(i)·적합도(u,i))` — N은 고객 간 학습 배분, V는 고객 내 상품 우선순위

무효 입력 행은 두 arm 모두 원시가중 1, 학습행 평균 1로 정규화. 처음부터 한 모델·한 optimizer로 공동학습하며 동결·점수 후처리 없음.

고정: Dunnhumby 역사적 개발 684–690일, seed 43, 300 epoch(25마다 기록), **판정은 300 epoch 고정**, 100은 진단. M1·M3(두 축)은 기존 3시드 M3 원본 JSON에서 재사용(재학습 없음). test/holdout은 열지 않습니다.

판정(실행 전 고정): ① M1 대비 가격·구매금액 가중 적중값@10과 가중 NDCG@10 모두 상승 ② 여섯 정확도 ≥ M1의 99% ③ 경제2지표@10이 M3의 99% 이상. 단일 개발시드이므로 유의성·일반화는 주장하지 않습니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
OLD_M3_DIR = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1'
if not list(OLD_M3_DIR.glob('clv_m3_centered_value_graph_*.json')):
    if not os.path.ismount('/content/drive'):
        try:
            drive.mount('/content/drive')
        except (ValueError, NotImplementedError) as exc:
            raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc
    if not list(OLD_M3_DIR.glob('clv_m3_centered_value_graph_*.json')):
        raise RuntimeError('M1·기존 N/V M3의 완료 결과가 없습니다. 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = '578c99fc7895127a043620334a447465fc50f881'
REPO = Path('/content/clv-m5-m3-m4-split-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'lightgcn_clv_m3_centered_value_graph', 'clv_m5_m3_m4_split_nv_screen'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError(f'{name}이 다른 코드에서 로드됐습니다. 런타임을 다시 시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_split_nv_screen as screen
screen.self_test()
cfg = screen.configure(m1_result_dir=str(OLD_M3_DIR))
print(json.dumps({'source_commit': SOURCE_COMMIT, 'seed': cfg.seeds, 'epochs': cfg.epochs,
                  'lambda': screen.LAMBDA, 'arms': [screen.ARM_A, screen.ARM_B], 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## 실행

학습 전에 ① 그래프 β가 기존 M3와 같은지 ② 기존 M1·M3 원본 JSON이 있는지 ③ A의 가중치가 기존 원형 M4 감사값(학습행 2,478,857, 평균 원시가중 1.14168269, CV 0.11679748)과 같은지 확인하고, 하나라도 다르면 학습하지 않고 멈춥니다. 새 GPU 학습은 A·B 두 arm(각 약 1.7시간). epoch마다 Drive에 저장되므로 끊기면 같은 셀을 다시 실행하면 이어집니다.

In [ ]:
result = screen.run(cfg)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
key = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
       'price_purchase_amount_weighted_hit@10', 'vndcg@10',
       'price_purchase_amount_weighted_hit@20', 'vndcg@20',
       'price_purchase_amount_weighted_hit@50', 'vndcg@50']
view = result['comparison']
view = view[view.metric.isin(key)]
for (epoch, model, ref), part in view.groupby(['epoch', 'model_id', 'reference']):
    print(f'\n=== epoch {epoch} | {model} vs {ref} ===')
    print(part[['metric', 'reference_value', 'candidate_value', 'ratio']].to_string(index=False))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
zip_path = Path('/content/clv_m5_m3_m4_split_nv_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))